# 02 · Обучение регрессоров и оценка тревог

Этот ноутбук готовит данные, обучает и постепенно обновляет детекторы, калибрует их пороги,
а затем показывает результаты **здесь, в ноутбуке**. Классификаторы в нём не обучаются.
Для классификатора по исходным датчикам используется `05_Классификатор_без_регрессора.ipynb`.

Выполняйте разделы сверху вниз. Чтобы посмотреть уже сохранённый расчёт, достаточно разделов **1, 2 и 4–6**:
ячейку обучения в разделе 3 можно пропустить. Переменные `RESULT` и `LIVE` для просмотра не нужны.
Блоки котлов и большие таблицы открываются стрелками. Изображения встроены как PNG, поэтому видны и после сохранения ноутбука.

Предыдущий 02 сохранён отдельно в `02_Обучение_до_переработки_2026-10-28.ipynb`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import os, sys, json, inspect
import pandas as pd
from IPython.display import display, HTML

_candidates = [Path.cwd(), Path.cwd()/"training"]
TRAINING_DIR = next((p.resolve() for p in _candidates if (p/"lib/monitoring_v3").is_dir()), None)
if TRAINING_DIR is None:
    raise FileNotFoundError("Откройте ноутбук из папки проекта или training.")
os.chdir(TRAINING_DIR)
for _path in (TRAINING_DIR, TRAINING_DIR/"lib"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

from notebook_reports import notebook_config
from monitoring_v3.data import load_sources
import monitoring_v3.pipeline as LF
print("Python:", sys.executable)
print("Папка проекта:", TRAINING_DIR)

## 1. Настройки

`SETTINGS` переопределяет `incremental_settings.json`; `SENSOR_RULES` — правила датчиков.
Здесь сохраняются прежние пути и группы, если вы их уже указывали в 02.

Основные настройки обучения:

* `models`: список моделей, например `["CatBoost"]`.
* `feature_sets`: `["conditions"]`, `["related"]`, `["forecast"]` или несколько вариантов.
  Conditions использует нагрузку и условия; related дополнительно использует другие датчики;
  forecast — только прошлые измерения с горизонтом из `horizons`. Для related нужны проверенные связи `related_inputs`.
* `scenarios`: `["local"]` — отдельно по котлам, `["pooled"]` — общая модель группы; возможно несколько вариантов.
* `first_test`: начало основного отчётного периода. Месячные модели до него могут строиться для исторической проверки.
* `run_thresholds`: пороги **нагрузки**, не пороги тревог. Например `{"К-5": 140}`.
* `validate_candidate`: проверять новую модель перед принятием; `False` ускоряет обновление с отказом от этой проверки.
* `output`: каталог расчёта. Для продолжения используйте прежний каталог; изменённые данные/параметры могут вызвать пересчёт.

Настройки просмотра `REPORT` не меняют обучение:

* `views`: схемы для подробных отчётов и графиков; `None` — все сохранённые схемы.
* `budgets`: бюджеты тревог из `alarm_budgets`; `None` — основной `alarm_budget`.
* `units`, `experiments`: списки для фильтрации; `None` — все выбранные в SETTINGS.
* `start`, `end`: даты просмотра; `None` — от first_test до конца рассчитанной истории. Метрики пересчитываются для выбранного периода.
* `sensors`: список датчиков для графика «факт / прогноз»; `None` — первые три.

В итоговую таблицу попадают все завершённые схемы и бюджеты, даже если подробные картинки ограничены `views`.

In [ ]:
SETTINGS = {'models': ['CatBoost'], 'feature_sets': ['conditions'], 'scenarios': ['local']}
# При необходимости добавьте сюда:
# "data_paths": {"К-5": "D:/private/К-5/hourly.parquet"},
# "groups": {"БКЗ-500": ["К-5", "К-6", "К-7"]},
# "feature_sets": ["conditions", "related"],
# "first_test": "2025-03-01", "output": "runs/my_experiment",
# "run_thresholds": {"К-5": 140}, "validate_candidate": False,

SENSOR_RULES = {'defaults': {}, 'sensors': {}}

REPORT = {
    "views": ["mean::instant", "top5::instant", "max::instant", "fraction::instant", "mean::slow48"],
    "budgets": None,
    "units": None, "experiments": None,
    "start": None, "end": None,
    "sensors": None,
}
CFG = notebook_config(SETTINGS, SENSOR_RULES)

### Служебный код вывода

Следующая ячейка содержит функции отчётов. Просто выполните её; изменять код не требуется.
В ней нет обучения моделей или классификаторов.

In [ ]:
import types
_DETECTOR_REPORT = types.ModuleType("_detector_report_02")
exec('"""Pandas / matplotlib / sklearn reports for detector notebook 02; no model fitting."""\nfrom pathlib import Path\nfrom html import escape\nfrom io import BytesIO\nimport base64\nimport json\nimport numpy as np\nimport pandas as pd\n\n\ndef read_csv(path):\n    path=Path(path)\n    if not path.exists():return pd.DataFrame()\n    try:return pd.read_csv(path)\n    except pd.errors.EmptyDataError:return pd.DataFrame()\n\n\ndef details(title,body,opened=False):\n    return \'<details\'+(\' open\' if opened else \'\')+\'><summary><b>\'+escape(str(title))+\'</b></summary>\'+body+\'</details>\'\n\n\ndef table(title,frame,opened=False):\n    body=(\'<p>Нет записей.</p>\' if frame.empty else\n          \'<div style="overflow:auto;max-height:520px">\'+frame.to_html(index=False,escape=True,na_rep=\'—\',max_rows=None,max_cols=None)+\'</div>\')\n    return details(title,body,opened)\n\n\ndef figure_html(fig,path):\n    import matplotlib.pyplot as plt\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    buffer=BytesIO()\n    try:\n        fig.savefig(buffer,format=\'png\',dpi=110,bbox_inches=\'tight\')\n        payload=buffer.getvalue();path.write_bytes(payload)\n        return \'<img style="max-width:100%;height:auto" src="data:image/png;base64,\'+base64.b64encode(payload).decode()+\'">\'\n    finally:plt.close(fig)\n\n\ndef show_html(body):\n    from IPython.display import HTML,display\n    display(HTML(body))\n\n\ndef preparation_report(contexts,issues,cfg):\n    from incremental_plots import plot_preparation\n    from monitoring_v3.pipeline import dataset\n    from monitoring_v3.data import healthy\n    from notebook_reports import check_preparation_thresholds\n    check_preparation_thresholds(contexts,cfg)\n    root=Path(cfg[\'output\'])/\'notebook_report\';root.mkdir(parents=True,exist_ok=True)\n    summary=[];events=[]\n    for unit,c in contexts.items():\n        frame=c[\'cleaned\'];ix=frame.index\n        summary.append(dict(unit=unit,start=ix[0],end=ix[-1],rows=len(ix),\n            sensors=len(c[\'targets\']),condition_features=c[\'features\'].shape[1],\n            eligible_rows=int(c[\'eligible\'].sum()),healthy_before_test=int(healthy(c,cfg[\'first_test\'],cfg).sum()),\n            rows_in_test=int((ix>=pd.Timestamp(cfg[\'first_test\'])).sum()),load=c[\'load\'],run_threshold=c[\'threshold\']))\n        e=c[\'events\'].copy();e.insert(0,\'unit\',unit);events.append(e)\n    summary=pd.DataFrame(summary)\n    summary.to_csv(root/\'data_summary.csv\',index=False,encoding=\'utf-8-sig\')\n    body=table(\'Источники и доступные данные\',summary,True)\n    if issues:body+=table(\'Проблемы источников\',pd.DataFrame(issues),True)\n    body+=\'<p>healthy_before_test — допустимая норма до начала проверки, до разделения на обучение и калибровку. \'\n    body+=\'Число строк конкретного месячного обучения показано далее в журнале отбора. \'\n    body+=\'Таблицы ниже разворачиваются стрелкой; признаки не обрезаются многоточием.</p>\'\n    for unit,c in contexts.items():\n        d=c[\'cleaned\'];cols=list(dict.fromkeys(c[\'targets\']+[k for k,r in c[\'roles\'].items() if r in (\'condition\',\'load\')]))\n        numeric=d[cols].apply(pd.to_numeric,errors=\'coerce\')\n        stats=numeric.describe().T.rename_axis(\'sensor\').reset_index()\n        stats[\'missing_fraction\']=numeric.isna().mean().to_numpy()\n        stats[\'unique_values\']=numeric.nunique().to_numpy()\n        for flag in (\'spike\',\'frozen\',\'masked\'):\n            stats[flag+\'_rows\']=c[\'quality\'].xs(flag,axis=1,level=1).sum().reindex(cols).to_numpy()\n        schema=pd.DataFrame([dict(sensor=k,role=v,unit=c.get(\'declared_units\',{}).get(k,\'\'),dtype=str(d[k].dtype),\n                                 unique_values=int(d[k].nunique()),missing_fraction=float(d[k].isna().mean()))\n                             for k,v in c[\'roles\'].items() if k in d])\n        part=table(\'Назначение и типы каналов\',schema)+table(\'Статистика и качество измерений\',stats)\n        part+=table(\'Пример исходных данных\',c[\'frame\'].head(5).reset_index())\n        part+=table(\'Пример после подготовки\',d.head(5).reset_index())\n        part+=table(\'Режимы: число строк\',c[\'regime\'][\'mode\'].value_counts().rename_axis(\'mode\').reset_index(name=\'rows\'),True)\n        part+=table(\'Реестр остановов и ремонтов\',c[\'events\'])\n        part+=table(\'Токи: сопоставление и пороги\',pd.DataFrame(c.get(\'regime_spec\',{}).get(\'audit\',[])))\n        if \'audit\' in c:part+=table(\'Сопоставление датчиков\',c[\'audit\'])\n        for variant in cfg[\'feature_sets\']:\n            horizons=cfg[\'horizons\'] if variant==\'forecast\' else [0]\n            for horizon in horizons:\n                x,y=dataset(c,variant,horizon,cfg)\n                names=pd.DataFrame({\'feature\':x.columns,\'dtype\':x.dtypes.astype(str).to_numpy(),\n                                    \'missing_fraction\':x.isna().mean().to_numpy()})\n                preview=y.index[y.notna().any(axis=1)][:5]\n                part+=details(f\'{variant}, горизонт {horizon} ч: {x.shape[1]} входных признаков; {y.shape[1]} целей\',\n                    \'<p>Для related это общий набор кандидатов: собственный датчик исключается из входов его модели. \'\n                    \'В pooled/transfer набор дополнительно согласуется между котлами.</p>\'+\n                    table(\'Все признаки\',names)+table(\'Пример X: первые допустимые строки\',x.loc[preview].reset_index())+\n                    table(\'Пример y для этих же строк\',y.loc[preview].reset_index()))\n                del x,y\n        part+=figure_html(plot_preparation(c),root/unit/\'preparation.png\')\n        body+=details(unit+\' · подготовка данных\',part)\n    event_frame=pd.concat(events,ignore_index=True) if events else pd.DataFrame()\n    body+=table(\'Все события\',event_frame)\n    (root/\'preparation.html\').write_text(body,encoding=\'utf-8\');show_html(body)\n    return summary\n\n\ndef saved_experiments(cfg):\n    from monitoring_v3.pipeline import experiments\n    planned=list(experiments(cfg));rows=[]\n    for meta in sorted((Path(cfg[\'output\'])/\'experiments\').glob(\'*/experiment.json\')):\n        spec=json.loads(meta.read_text(encoding=\'utf-8\'))\n        if not any(spec==s for s in planned):continue\n        for unit in spec[\'recipients\']:\n            rows.append(dict(experiment=meta.parent.name,unit=unit,method=spec[\'method\'],variant=spec[\'variant\'],\n                             scenario=spec[\'scenario\'],directory=meta.parent/unit,spec=spec))\n    return rows\n\n\ndef load_timeline(item,view,budget,cfg,context):\n    """Read finalized output, or reconstruct an explicitly provisional interrupted run.\n\n    Only cached scores with the current checkpoint/data fingerprint are admitted.\n    No fitting or retrospective prediction using the latest model is performed.\n    """\n    from monitoring_v3.alarm_scores import alarm_stream,view_policy\n    import joblib\n    directory=Path(item[\'directory\']);spec=item[\'spec\'];unit=item[\'unit\']\n    slug=view.replace(\':\',\'_\')+\'__\'+str(float(budget))\n    path=directory/(slug+\'.parquet\')\n    candidates_on_disk=(list((directory.parent/\'scored\').glob(\'*/\'+unit+\'.joblib\'))+\n                        list((directory.parent/\'scored\').glob(\'*/\'+unit+\'.detector.joblib\')))\n    if path.exists() and not any(p.stat().st_mtime_ns>path.stat().st_mtime_ns for p in candidates_on_disk):\n        return pd.read_parquet(path),False\n    from monitoring_v3.pipeline import score_fingerprint\n    from monitoring_v3.checkpoint_cache import semantic_config\n    # Checkpoint compatibility is deliberately required for partial results.\n    # Full exports have already been written, and remain useful independently.\n    parts=[]\n    for checkpoint in sorted((directory.parent/\'checkpoints\').glob(\'*.joblib\')):\n        folder=directory.parent/\'scored\'/checkpoint.stem\n        candidates=[p for p in (folder/(unit+\'.detector.joblib\'),folder/(unit+\'.joblib\')) if p.exists()]\n        if not candidates:continue\n        end=pd.Timestamp(checkpoint.stem)+pd.offsets.MonthBegin(1)\n        month=None\n        for cached in sorted(candidates,key=lambda p:p.stat().st_mtime_ns,reverse=True):\n            value=joblib.load(cached)\n            token=value.get(\'model_token\');old_cfg=value.get(\'report_config\')\n            if token is None or old_cfg is None:\n                # Compatibility only: modern score caches carry these small\n                # signatures, so viewing them never loads 226 native models.\n                saved_state=joblib.load(checkpoint)\n                token=saved_state.get(\'compatibility\')\n                old_cfg=getattr(saved_state[\'state\'][\'model\'],\'cfg\',{})\n                del saved_state\n            if token is None or semantic_config(old_cfg)!=semantic_config(cfg):continue\n            expected=score_fingerprint(context,end,token)\n            if value.get(\'fingerprint\')==expected:month=value;break\n        if month is None:continue\n        scores=month[\'month\'][\'views\'].get(view)\n        if scores is not None:parts.append(scores)\n    if not parts:return pd.DataFrame(),True\n    d=pd.concat(parts).sort_index();d=d.loc[~d.index.duplicated(keep=\'last\')]\n    first=min(pd.Timestamp(cfg[\'first_test\']),d.index[0])\n    ix=context[\'frame\'].index\n    ix=ix[(ix>=first)&(ix<=d.index[-1])];d=d.reindex(ix)\n    threshold=\'threshold_budget_\'+str(float(budget))\n    if threshold not in d:raise ValueError(\'Нет сохранённого порога \'+threshold)\n    d[\'ratio\']=d.smoothed/d[threshold].where(np.isfinite(d[threshold]))\n    stream,_=alarm_stream(d.ratio,view_policy(cfg,view))\n    d[\'active\']=stream.active;d[\'notification\']=stream.notification\n    d[\'eligible\']=context[\'eligible\'].reindex(ix).fillna(False)\n    d[\'background_eligible\']=context[\'frame\'][\'Состояние\'].reindex(ix).eq(\'Работа\')\n    d[\'load_observed\']=context[\'cleaned\'][context[\'load\']].reindex(ix).notna()\n    return d,True\n\n\ndef hourly_examples(timeline,events,cfg,start,end=None):\n    """Exactly the hourly ranking population used by alarm_scores.evaluate.\n\n    Positive = an eligible score in the configured pre-failure window; negative\n    = eligible observed background. Neutral/unknown intervals are excluded.\n    This is not a classifier ground truth for arbitrary types of anomaly.\n    """\n    ix=timeline.index\n    end=pd.Timestamp(end) if end is not None else ix[-1]+pd.Timedelta(cfg[\'step\'])\n    valid=timeline.score.notna()&np.isfinite(timeline.ratio)&(ix>=pd.Timestamp(start))&(ix<end)\n    positive=pd.Series(False,index=ix);background=pd.Series(True,index=ix);uncertain=positive.copy()\n    for e in events.itertuples():\n        if pd.isna(e.stop):continue\n        if e.type==\'Неизвестно\':uncertain|=(ix>=e.stop-pd.Timedelta(days=cfg[\'pre_window_days\']))&(ix<e.stop)\n        if e.type!=\'Авария\':continue\n        positive|=(ix>=e.stop-pd.Timedelta(days=cfg[\'pre_window_days\']))&(ix<e.stop-pd.Timedelta(hours=cfg[\'min_lead_hours\']))\n        background&=~((ix>=e.stop-pd.Timedelta(days=cfg[\'pre_window_days\']+cfg[\'neutral_days\']))&(ix<e.stop))\n    background&=valid&~uncertain&timeline.get(\'background_eligible\',pd.Series(True,index=ix)).fillna(False)\n    take=(positive|background)&valid\n    return positive.loc[take].astype(int),timeline.loc[take,\'ratio\'],timeline.loc[take,\'active\'].fillna(False).astype(int)\n\n\ndef timeline_figure(timeline,context,cfg,title,start,end,budget=None):\n    import matplotlib.pyplot as plt\n    d=timeline.loc[start:end]\n    fig,axes=plt.subplots(3,1,figsize=(14,7),sharex=True,gridspec_kw={\'height_ratios\':[3,1,1]})\n    if d.empty:\n        axes[0].text(.5,.5,\'В выбранном периоде ещё нет результатов\',ha=\'center\',transform=axes[0].transAxes)\n        return fig\n    axes[0].plot(d.index,d.ratio,lw=.7,label=\'Отклонение / действующий порог\')\n    axes[0].axhline(1,color=\'red\',ls=\'--\',label=\'Порог тревоги\')\n    active=d.active.fillna(False).to_numpy(bool)\n    axes[0].fill_between(d.index,0,1,where=active,transform=axes[0].get_xaxis_transform(),color=\'red\',alpha=.12,label=\'Тревога\')\n    notices=d.loc[d.notification.fillna(False)]\n    axes[0].scatter(notices.index,notices.ratio,s=12,color=\'red\',label=\'Уведомление\')\n    axes[0].set_yscale(\'symlog\',linthresh=1);axes[0].set_ylabel(\'Отклонение / порог\');axes[0].set_title(title)\n    key=\'threshold_budget_\'+str(float(budget if budget is not None else cfg[\'alarm_budget\']))\n    threshold=d[key] if key in d else d[\'smoothed\'].div(d.ratio).replace([np.inf,-np.inf],np.nan)\n    axes[1].plot(d.index,d.smoothed,lw=.7,label=\'Сглаженное отклонение\')\n    axes[1].plot(d.index,threshold,lw=1,color=\'red\',ls=\'--\',label=\'Фактический порог этой схемы\')\n    axes[1].set_yscale(\'symlog\',linthresh=1);axes[1].set_ylabel(\'Балл\')\n    mode=context[\'regime\'][\'mode\'].reindex(d.index)\n    axes[2].step(d.index,mode.map({\'стоит\':0,\'неизвестно\':1,\'пуск\':2,\'работа\':3}),where=\'post\',lw=.8)\n    axes[2].set_yticks([0,1,2,3],[\'стоит\',\'неизвестно\',\'пуск\',\'работа\']);axes[2].set_ylabel(\'Режим\')\n    for e in context[\'events\'].itertuples():\n        if pd.isna(e.start) or pd.isna(e.end) or e.start>d.index[-1] or e.end<d.index[0]:continue\n        for ax in axes:ax.axvspan(e.start,e.end,color=\'red\' if e.type==\'Авария\' else \'gray\',alpha=.12)\n    for ax in axes:ax.grid(alpha=.2)\n    axes[0].legend(fontsize=8,loc=\'upper left\');axes[1].legend(fontsize=8)\n    fig.tight_layout();return fig\n\n\ndef regression_table(directory,context=None,start=None,end=None):\n    d=read_csv(Path(directory)/\'prediction_diagnostics.csv\')\n    if \'month\' in d:\n        when=pd.to_datetime(d.month)\n        if start is not None:d=d.loc[when.ge(pd.Timestamp(start).to_period(\'M\').to_timestamp())]\n        if end is not None:d=d.loc[pd.to_datetime(d.month).le(pd.Timestamp(end))]\n    predictions=Path(directory)/\'prequential_predictions.parquet\'\n    errors=Path(directory)/\'prequential_errors.parquet\'\n    if context is not None and predictions.exists() and errors.exists():\n        from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score\n        predicted=pd.read_parquet(predictions).loc[start:end]\n        valid=pd.read_parquet(errors).reindex(predicted.index).notna()\n        rows=[]\n        for sensor in predicted:\n            if sensor not in context[\'cleaned\']:continue\n            a=context[\'cleaned\'][sensor].reindex(predicted.index)\n            p=predicted[sensor]\n            take=valid[sensor]&np.isfinite(a)&np.isfinite(p)\n            a,p=a.loc[take].astype(float),p.loc[take].astype(float)\n            if not len(a):continue\n            rows.append(dict(sensor=sensor,rows=len(a),MAE=float(mean_absolute_error(a,p)),\n                RMSE=float(np.sqrt(mean_squared_error(a,p))),\n                R2=float(r2_score(a,p)) if len(a)>1 and a.nunique()>1 else np.nan))\n        return pd.DataFrame(rows),d\n    sufficient=[\'regression_rows\',\'absolute_error_sum\',\'squared_error_sum\',\'target_sum\',\'target_squared_sum\']\n    if d.empty or not set(sufficient).issubset(d):return pd.DataFrame(),d\n    d=d.loc[d.regression_rows.gt(0)].copy()\n    grouped=d.groupby(\'sensor\')[sufficient].sum()\n    n=grouped.regression_rows\n    out=pd.DataFrame({\'sensor\':grouped.index,\'rows\':n.to_numpy(),\n        \'MAE\':(grouped.absolute_error_sum/n).to_numpy(),\n        \'RMSE\':np.sqrt(grouped.squared_error_sum/n).to_numpy()})\n    total=grouped.target_squared_sum-grouped.target_sum**2/n\n    out[\'R2\']=(1-grouped.squared_error_sum/total.where(total.gt(1e-10))).to_numpy()\n    return out,d\n\n\ndef classification_html(y,score,pred,path,prediction_title=\'Подтверждённая тревога, почасовая оценка\'):\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import (classification_report,ConfusionMatrixDisplay,roc_auc_score,\n                                average_precision_score,RocCurveDisplay,PrecisionRecallDisplay)\n    if not len(y):return \'<p>Нет часов с известной оценочной меткой.</p>\',{}\n    two=y.nunique()==2\n    roc=float(roc_auc_score(y,score)) if two else np.nan\n    ap=float(average_precision_score(y,score)) if two else np.nan\n    chance=float(y.mean())\n    metrics={\'hourly_roc_auc\':roc,\'hourly_average_precision\':ap,\'hourly_random_ap\':chance,\n             \'hourly_rows\':len(y),\'hourly_positive_rows\':int(y.sum())}\n    from sklearn.metrics import precision_recall_fscore_support\n    precision,recall,f1,_=precision_recall_fscore_support(y,pred,labels=[1],zero_division=0)\n    metrics.update(hourly_precision=float(precision[0]),hourly_recall=float(recall[0]),hourly_f1=float(f1[0]))\n    fmt=lambda v:\'—\' if not np.isfinite(v) else f\'{v:.4f}\'\n    body=f\'<p><b>ROC AUC: {fmt(roc)}; Average Precision: {fmt(ap)}; случайный AP: {chance:.4f}; случайный ROC AUC: 0.5.</b></p>\'\n    if not two:body+=\'<p>В выбранном периоде один класс: ROC AUC и AP не оцениваем.</p>\'\n    body+=\'<pre>\'+escape(classification_report(y,pred,labels=[0,1],target_names=[\'фон\',\'перед аварией\'],digits=3,zero_division=0))+\'</pre>\'\n    fig,ax=plt.subplots(figsize=(5,4));ConfusionMatrixDisplay.from_predictions(y,pred,labels=[0,1],\n        display_labels=[\'фон\',\'перед аварией\'],ax=ax,colorbar=False,cmap=\'Blues\',values_format=\'d\')\n    ax.set_title(prediction_title)\n    body+=figure_html(fig,Path(path)/\'confusion_matrix.png\')\n    if two:\n        fig,axes=plt.subplots(1,2,figsize=(11,4))\n        RocCurveDisplay.from_predictions(y,score,ax=axes[0]);axes[0].plot([0,1],[0,1],\'--\',color=\'gray\',label=\'Случайно\')\n        PrecisionRecallDisplay.from_predictions(y,score,ax=axes[1]);axes[1].axhline(chance,color=\'gray\',ls=\'--\',label=\'Случайный AP\')\n        for ax in axes:ax.legend(fontsize=8)\n        body+=details(\'ROC и Precision–Recall\',figure_html(fig,Path(path)/\'ranking_curves.png\'))\n    return body,metrics\n\n\ndef prediction_plot(directory,context,start,end,sensors=None):\n    import matplotlib.pyplot as plt\n    path=Path(directory)/\'prequential_predictions.parquet\'\n    if not path.exists():return (\'<p>Старый расчёт не сохранял сами прогнозы. Таблицы и графики тревог доступны; \'\n        \'график факта и прогноза появится после расчёта новой версией.</p>\')\n    predicted=pd.read_parquet(path).loc[start:end]\n    chosen=[s for s in (sensors or list(predicted.columns[:3])) if s in predicted and s in context[\'cleaned\']]\n    if not chosen or predicted.empty:return \'<p>Нет прогнозов для выбранных датчиков и периода.</p>\'\n    fig,axes=plt.subplots(len(chosen),1,figsize=(14,3*len(chosen)),squeeze=False,sharex=True)\n    for ax,sensor in zip(axes[:,0],chosen):\n        actual=context[\'cleaned\'][sensor].reindex(predicted.index)\n        ax.plot(actual.index,actual,lw=.7,label=\'Измерение\')\n        ax.plot(predicted.index,predicted[sensor],lw=.8,label=\'Прогноз модели, доступной в этот месяц\')\n        ax.set_title(sensor);ax.set_ylabel(context.get(\'declared_units\',{}).get(sensor,\'\'));ax.legend(fontsize=8);ax.grid(alpha=.2)\n    fig.tight_layout();return figure_html(fig,Path(directory)/\'notebook_report\'/\'actual_prediction.png\')\n\n\ndef render_detector_results(cfg,contexts,options=None):\n    """Rebuild all inline output from saved monthly / final results, no RESULT needed."""\n    from monitoring_v3.alarm_scores import evaluate,view_policy\n    options={\'views\':[\'mean::instant\',\'top5::instant\',\'max::instant\',\'fraction::instant\',\'mean::slow48\'],\n             \'budgets\':None,\'start\':None,\'end\':None,\'units\':None,\'experiments\':None,\'sensors\':None,**(options or {})}\n    root=Path(cfg[\'output\']);(root/\'notebook_report\').mkdir(parents=True,exist_ok=True)\n    items=saved_experiments(cfg);rows=[];regression_rows=[]\n    for item in items:\n        if options[\'units\'] and item[\'unit\'] not in options[\'units\']:continue\n        if options[\'experiments\'] and item[\'experiment\'] not in options[\'experiments\']:continue\n        directory=Path(item[\'directory\']);directory.mkdir(parents=True,exist_ok=True)\n        c=contexts[item[\'unit\']];report_dir=directory/\'notebook_report\';report_dir.mkdir(exist_ok=True)\n        if item[\'scenario\']!=\'local\':\n            from monitoring_v3.pipeline import align_contexts,transfer_preparation\n            members=list(dict.fromkeys(item[\'spec\'][\'sources\']+item[\'spec\'][\'recipients\']))\n            c=transfer_preparation(align_contexts(contexts,members),item[\'spec\'],cfg)[item[\'unit\']]\n        title=item[\'method\']+\' · \'+item[\'unit\']+\' · \'+item[\'variant\']+\' · \'+item[\'scenario\']\n        part=\'<p><b>\'+escape(title)+\'</b></p>\'\n        part+=\'<p>Модель для дашборда: <code>\'+escape(str((directory/\'model\').resolve()))+\'</code></p>\'\n        updates=directory.parent/\'updates.json\'\n        if updates.exists():\n            update_frame=pd.DataFrame(json.loads(updates.read_text(encoding=\'utf-8\')))\n            skip=[\'trees\',\'target_errors\',\'calibration_by_unit\',\'calibration_sensor_diagnostics\',\'admission_by_unit\']\n            part+=table(\'Все месяцы: обучение, обновление, причины пропуска\',update_frame.drop(columns=skip,errors=\'ignore\'))\n        for filename,label in [(\'admission_diagnostics.csv\',\'Отбор часов: куда ушла история\'),\n                               (\'calibration_diagnostics.csv\',\'Калибровка: периоды и пороги\'),\n                               (\'target_update_diagnostics.csv\',\'Принятие обновлений по датчикам\')]:\n            frame=read_csv(directory.parent/filename)\n            if \'unit\' in frame:frame=frame.loc[frame.unit.eq(item[\'unit\'])]\n            if len(frame):part+=table(label,frame)\n        regression,diagnostics=regression_table(directory,c,options[\'start\'] or cfg[\'first_test\'],options[\'end\'])\n        if len(regression):\n            regression[\'unit\']=regression.sensor.map(c.get(\'declared_units\',{})).fillna(\'\')\n            regression_rows.append(regression.rename(columns={\'unit\':\'measurement_unit\'}).assign(\n                experiment=item[\'experiment\'],unit=item[\'unit\'],variant=item[\'variant\'],method=item[\'method\']))\n            regression.to_csv(report_dir/\'regression_metrics.csv\',index=False,encoding=\'utf-8-sig\')\n            part+=table(\'Регрессия на последующих месяцах: MAE, RMSE, R² по датчикам\',regression)\n            part+=\'<p>MAE и RMSE — в единицах датчика, только на часах с допустимой диагностикой и известным измерением. Разные датчики в одну физическую ошибку не усредняем. \'\n            part+=\'R² сравнивает с постоянным средним; отрицательное значение означает хуже этой константы на периоде оценки.</p>\'\n        elif len(diagnostics):part+=\'<p>Этот сохранённый расчёт содержит старые диагностические метрики, но не MAE/RMSE/R².</p>\'\n        if len(diagnostics):part+=table(\'Регрессия: все датчики и месяцы\',diagnostics.drop(columns=[\n            \'absolute_error_sum\',\'squared_error_sum\',\'target_sum\',\'target_squared_sum\'],errors=\'ignore\'))\n        importance=read_csv(directory.parent/\'feature_importance_summary.csv\')\n        if len(importance) and \'mean_importance_percent\' in importance:\n            importance=importance.loc[pd.to_numeric(importance.mean_importance_percent,errors=\'coerce\').fillna(0).ne(0)]\n            part+=table(\'Важность: только ненулевые входы, среднее по моделям датчиков\',importance)\n            if len(importance):\n                import matplotlib.pyplot as plt\n                top=importance.nlargest(20,\'mean_importance_percent\').iloc[::-1]\n                fig,ax=plt.subplots(figsize=(12,max(4,.28*len(top))))\n                ax.barh(top.feature,top.mean_importance_percent);ax.set_xlabel(\'Средняя важность, %\');fig.tight_layout()\n                part+=details(\'20 наиболее важных входов\',figure_html(fig,report_dir/\'importance.png\'))\n        detailed_views=options[\'views\']\n        # Completed output includes every available scheme; partial output uses\n        # the requested schemes only and is explicitly marked below.\n        views=list(detailed_views or [])\n        if detailed_views is None:\n            def decode(stem):\n                name,mode=stem.rsplit(\'__\',1)[0].rsplit(\'__\',1)\n                return (name.replace(\'group_\',\'group:\',1) if name.startswith(\'group_\') else name)+\'::\'+mode\n            views=[decode(p.stem) for p in directory.glob(\'*__*.parquet\')]\n            views=list(dict.fromkeys(views)) or [\'mean::instant\']\n        # Summary covers ALL saved schemes, even when detailed figures are limited.\n        summary=read_csv(root/\'comparison.csv\')\n        existing=summary.loc[summary.experiment.eq(item[\'experiment\'])&summary.unit.eq(item[\'unit\'])] if {\'experiment\',\'unit\'}.issubset(summary) else pd.DataFrame()\n        all_pairs={(str(row.view),float(row.budget)) for row in existing.itertuples()}\n        budgets=options[\'budgets\'] if options[\'budgets\'] is not None else [float(cfg[\'alarm_budget\'])]\n        pairs=sorted(all_pairs|{(v,float(b)) for v in views for b in budgets})\n        for view,budget in pairs:\n            timeline,partial=load_timeline(item,view,budget,cfg,c)\n            if timeline.empty:continue\n            left=pd.Timestamp(options[\'start\'] or cfg[\'first_test\']);right=pd.Timestamp(options[\'end\']) if options[\'end\'] else None\n            if not len(timeline.loc[left:right]):continue\n            policy=view_policy(cfg,view)\n            # Figure ends are inclusive pandas timestamps; evaluation is half-open.\n            eval_end=right+pd.Timedelta(cfg[\'step\']) if right is not None else None\n            metrics,events=evaluate(timeline,c[\'events\'],policy,start=left,end=eval_end)\n            y,score,pred=hourly_examples(timeline,c[\'events\'],policy,left,eval_end)\n            from sklearn.metrics import roc_auc_score,average_precision_score,precision_recall_fscore_support\n            two=y.nunique()==2\n            hp,hr,hf,_=precision_recall_fscore_support(y,pred,labels=[1],zero_division=0) if len(y) else ([np.nan],[np.nan],[np.nan],None)\n            hm=dict(hourly_roc_auc=float(roc_auc_score(y,score)) if two else np.nan,\n                    hourly_average_precision=float(average_precision_score(y,score)) if two else np.nan,\n                    hourly_random_ap=float(y.mean()) if len(y) else np.nan,\n                    hourly_rows=len(y),hourly_positive_rows=int(y.sum()),hourly_precision=float(hp[0]),hourly_recall=float(hr[0]),hourly_f1=float(hf[0]))\n            rows.append(dict(experiment=item[\'experiment\'],unit=item[\'unit\'],method=item[\'method\'],variant=item[\'variant\'],\n                view=view,budget=budget,partial=partial,**metrics,**hm))\n            if view not in views or budget not in budgets:continue\n            slug=view.replace(\':\',\'_\')+\'__\'+str(budget);out=report_dir/slug\n            report,hm=classification_html(y,score,pred,out)\n            subset=pd.DataFrame([dict(view=view,budget=budget,**metrics)])\n            if partial:report=\'<p><b>Промежуточный результат: эксперимент был остановлен; используются только готовые месячные оценки.</b></p>\'+report\n            report=table(\'Аварии и уведомления (отдельно от почасового classification report)\',subset,True)+report\n            report+=figure_html(timeline_figure(timeline,c,cfg,title+\' · \'+view,left,right,budget),out/\'timeline.png\')\n            report+=table(\'Какие аварии обнаружены\',events)\n            from monitoring_v3.alarm_scores import alarm_stream\n            _,episodes=alarm_stream(timeline.ratio,policy)\n            report+=table(\'Все эпизоды тревоги\',episodes)\n            if len(y):pd.DataFrame({\'label\':y,\'score\':score,\'predicted_alarm\':pred}).to_parquet(out/\'hourly_evaluation.parquet\')\n            part+=details(view+f\' · бюджет {budget}/720 ч\',report,view==\'mean::instant\')\n        part+=details(\'Измерения и прогнозы датчиков (модель своего месяца)\',prediction_plot(directory,c,\n            options[\'start\'] or cfg[\'first_test\'],options[\'end\'],options[\'sensors\']))\n        final=details(title,part)\n        (report_dir/\'report.html\').write_text(final,encoding=\'utf-8\');show_html(final)\n    result=pd.DataFrame(rows)\n    if result.empty:\n        show_html(\'<p>Готовых оценок выбранных экспериментов нет. Проверьте output, выбор моделей и журнал месячных обновлений выше.</p>\')\n    else:\n        result.to_csv(root/\'notebook_report\'/\'all_metrics.csv\',index=False,encoding=\'utf-8-sig\')\n        preferred=[\'unit\',\'method\',\'variant\',\'view\',\'budget\',\'partial\',\'events\',\'detected\',\'recall\',\'precision\',\'average_precision\',\'random_precision\',\n                   \'hourly_roc_auc\',\'hourly_precision\',\'hourly_recall\',\'hourly_f1\',\'alarms_per_720h\',\'background_fraction\',\'lead_hours\',\'availability_operating\']\n        show_html(table(\'Итоговая таблица: котлы, модели и все рассчитанные схемы\',result.reindex(columns=preferred),len(result)<=30)+\n                  table(\'Все метрики, включая числители и исключённые часы/события\',result))\n    if regression_rows:\n        combined=pd.concat(regression_rows,ignore_index=True)\n        combined.to_csv(root/\'notebook_report\'/\'regressor_sensor_metrics.csv\',index=False,encoding=\'utf-8-sig\')\n        show_html(table(\'Все котлы и датчики: MAE, RMSE, R² за выбранный период\',combined))\n    return result\n\n\ndef saved_classifier_report(options,output):\n    """Inspect saved out-of-time 05 predictions. Never load or fit a model."""\n    import matplotlib.pyplot as plt\n    root=Path(options.get(\'root\',\'runs/direct_classifier_comparison\'))\n    explicit=options.get(\'file\')\n    paths=[Path(explicit)] if explicit else sorted(root.rglob(\'test_predictions.parquet\')) if root.exists() else []\n    paths=[p for p in paths if p.exists()]\n    if options.get(\'folds\',\'latest\')==\'latest\' and not explicit:\n        latest={}\n        for p in paths:\n            key=p.parent.parent if p.parent.name.startswith(\'fold_\') else p.parent\n            if key not in latest or p.parent.name>latest[key].parent.name:latest[key]=p\n        paths=list(latest.values())\n    if not paths:\n        show_html(\'<p>Сохранённых test_predictions.parquet из 05 не найдено. Обучение здесь не запускается. \'\n                  \'Укажите CLASSIFIER_VIEW.root (выходной каталог 05) либо CLASSIFIER_VIEW.file (конкретный файл).</p>\')\n        return pd.DataFrame()\n    summary=[];out_root=Path(output)/\'notebook_report\'/\'classifier_inspection\'\n    for number,path in enumerate(paths,1):\n        d=pd.read_parquet(path)\n        if not {\'score\',\'label\',\'evaluation_eligible\'}.issubset(d):\n            show_html(\'<p>Неподходящий формат: \'+escape(str(path))+\'</p>\');continue\n        settings_path=path.parent/\'settings.json\'\n        settings=json.loads(settings_path.read_text(encoding=\'utf-8\')) if settings_path.exists() else {}\n        rule_name=options.get(\'threshold_rule\',\'best_f2\')\n        rules=settings.get(\'thresholds\') or {}\n        threshold=float(rules.get(rule_name,{}).get(\'threshold\',.5))\n        fallback=rule_name not in rules\n        rule=rules.get(rule_name,{})\n        if rule_name!=\'default_05\' and (fallback or not np.isfinite(threshold) or not rule.get(\'available\',True)\n                or any(s in rule.get(\'status\',\'\') for s in (\'fallback\',\'threshold_unavailable\',\'not_evaluable\'))):\n            show_html(\'<p>\'+escape(str(path))+\' · выбранный порог \'+escape(rule_name)+\n                \' не подобран. Для просмотра фиксированного baseline укажите threshold_rule="default_05". Подстановка 0.5 не выполняется.</p>\')\n            continue\n        units=d.index.get_level_values(\'unit\').unique() if isinstance(d.index,pd.MultiIndex) and \'unit\' in d.index.names else [\'all\']\n        for unit in units:\n            if options.get(\'units\') and unit not in options[\'units\']:continue\n            part=d.xs(unit,level=\'unit\').copy() if unit!=\'all\' else d.copy()\n            if isinstance(part.index,pd.MultiIndex):\n                part.index=pd.DatetimeIndex(part.index.get_level_values(\'time\'))\n            else:part.index=pd.to_datetime(part.index)\n            part=part.sort_index().loc[options.get(\'start\'):options.get(\'end\')]\n            if part.empty:continue\n            if \'model_available_at\' in part:\n                causal=pd.to_datetime(part.model_available_at).le(part.index)\n            elif settings.get(\'available_at\'):\n                causal=pd.Series(part.index>=pd.Timestamp(settings[\'available_at\']),index=part.index)\n            else:\n                show_html(\'<p>\'+escape(str(path))+\' · не указана дата доступности модели: независимость оценок не подтверждена. \'\n                          \'Используйте тестовый экспорт актуального 05.</p>\');continue\n            if not causal.all():\n                show_html(\'<p>\'+escape(str(path))+\' · исключены оценки до даты доступности модели: \'+str(int((~causal).sum()))+\'</p>\')\n                part=part.loc[causal]\n            eligible=part.evaluation_eligible.eq(True)&part.label.isin([0,1])&np.isfinite(part.score)\n            tested=part.loc[eligible];y=tested.label.astype(int);score=tested.score\n            pred=score.ge(threshold).astype(int)\n            out=out_root/f\'model_{number:03d}\'/str(unit)\n            html,metrics=classification_html(y,score,pred,out,\'Классификатор 05: выбранный порог\')\n            html=\'<p>Источник: <code>\'+escape(str(path.resolve()))+\'</code></p>\'+html\n            html+=\'<p>Используются сохранённые test-оценки. Обучение и подбор порога здесь не выполняются. \'\n            html+=\'Положительный класс — предаварийные часы горизонта этой модели; остальные виды аномалий он не размечает.</p>\'\n            status=\'fallback 0.5: такого правила нет в сохранённых настройках\' if fallback else str(rules[rule_name].get(\'status\',\'сохранённый validation-порог\'))\n            html+=\'<p>Порог \'+escape(rule_name)+f\' = {threshold:.4f}; \'+escape(status)+\'.</p>\'\n            fig,axes=plt.subplots(2,1,figsize=(14,6),gridspec_kw={\'height_ratios\':[2,1]})\n            grid=pd.date_range(part.index[0],part.index[-1],freq=settings.get(\'alarm_step\',\'1h\'))\n            axes[0].plot(grid,part.score.reindex(grid),lw=.8,label=\'Балл классификатора\')\n            axes[0].axhline(threshold,color=\'red\',ls=\'--\',label=\'Сохранённый порог\')\n            axes[0].fill_between(part.index,0,1,where=(part.label.eq(1)&part.evaluation_eligible.eq(True)).to_numpy(bool),\n                transform=axes[0].get_xaxis_transform(),alpha=.15,color=\'red\',label=\'Размеченные предаварийные часы\')\n            axes[0].set_ylabel(\'Балл риска\');axes[0].legend(fontsize=8)\n            for label,title,color in ((0,\'Фон\',\'#3498db\'),(1,\'Перед аварией\',\'#e74c3c\')):\n                values=score.loc[y.eq(label)]\n                if len(values):axes[1].hist(values,bins=np.linspace(0,1,31),density=True,alpha=.5,label=title,color=color)\n            axes[1].axvline(threshold,color=\'red\',ls=\'--\');axes[1].set_xlabel(\'Балл риска\');axes[1].set_ylabel(\'Плотность\');axes[1].legend(fontsize=8)\n            fig.tight_layout();html+=figure_html(fig,out/\'normal_vs_pre_failure.png\')\n            html+=table(\'Сохранённые оценки всех часов выбранного участка\',part.reset_index())\n            show_html(details(str(unit)+\' · классификатор · \'+str(path.parent.relative_to(root)) if path.is_relative_to(root) else str(unit)+\' · \'+str(path.parent),html))\n            summary.append(dict(unit=unit,source=str(path),threshold_rule=rule_name,threshold=threshold,**metrics))\n    result=pd.DataFrame(summary)\n    if len(result):\n        out_root.mkdir(parents=True,exist_ok=True);result.to_csv(out_root/\'metrics.csv\',index=False,encoding=\'utf-8-sig\')\n        show_html(table(\'Классификаторы 05: просмотр готовых тестовых оценок\',result,len(result)<=20))\n    return result\n', _DETECTOR_REPORT.__dict__)
DR = _DETECTOR_REPORT
DR.show_html(DR.table("Действующие настройки", pd.DataFrame({"parameter": list(CFG), "value": [str(v) for v in CFG.values()]})))

## 2. Данные до обучения

Показываем размер и период истории, число каналов и признаков, пропуски, типы, примеры X и y,
режимы работы, фактический порог нагрузки и события. Большие таблицы скрыты по умолчанию.
Отметки ремонтов используются для отбора и оценки, **не подаются регрессору как признаки**.

In [ ]:
CFG = notebook_config(SETTINGS, SENSOR_RULES)
CONTEXTS, ISSUES = load_sources(CFG, Path(CFG["output"])/"audit", strict=False)
DATA_SUMMARY = DR.preparation_report(CONTEXTS, ISSUES, CFG)

## 3. Обучение и последовательная проверка

Для каждого месячного среза используются только сведения, доступные к его началу.
Допущенная прошлая норма разделяется на обучение, проверку кандидата (если включена) и калибровку.
Калибровка выбирает порог по ограничениям на частоту фоновых уведомлений и долю фона в тревоге;
это **не** поиск лучшего F1 на тестовых авариях. Следующий месяц оценивается уже зафиксированной моделью и порогом.
Затем история расширяется и предыдущая модель обновляется. При непригодном обновлении сохраняется предыдущая модель.

Существующий обновляемый индикатор показывает текущую операцию и датчик. После завершения запуска автоматически
выводятся отчёты раздела 4. Сохранённые модели доступны для дашборда после каждого месячного среза.
При остановке ячейки можно выполнить раздел 4 отдельно. Неполный расчёт помечается явно;
частичные оценки читаются из месячного кеша, если его подпись совпадает с checkpoint и исходными данными.

In [ ]:
if ISSUES:
    raise ValueError("Исправьте проблемы источников в SETTINGS и повторите раздел 2.")

TRAINING_RESULT = None
try:
    if "train_classifiers" in inspect.signature(LF.run).parameters:
        TRAINING_RESULT = LF.run(CFG, train_classifiers=False)
    else:
        # Совместимость со старой библиотекой: обучение риска также отключено.
        print("Используется старая библиотека. Для ускорения расчёта без признаков риска обновите training/lib.")
        TRAINING_RESULT = LF.run(CFG, interpretation={"run": False, "attach": False, "as_of": None})
except KeyboardInterrupt:
    print("Расчёт остановлен. Готовые модели сохранены. Для просмотра выполните раздел 4.")
else:
    METRICS = DR.render_detector_results(CFG, CONTEXTS, REPORT)

## 4. Все результаты в ноутбуке — повторный просмотр без обучения

Эта ячейка работает независимо от TRAINING_RESULT и LIVE. Она читает сохранённые результаты **всех** выбранных
котлов и вариантов модели. При первом запуске отчёты уже появились после обучения; повторять ячейку нужно для
другого периода/набора схем либо после остановки обучения. Чтобы посмотреть другой output, измените SETTINGS,
повторите разделы 1–2 и выполните эту ячейку.

В каждом сворачиваемом блоке: месячные решения и отбор часов; регрессионные метрики по датчикам;
важность ненулевых признаков; classification report и confusion matrix, ROC/PR;
графики отклонений, порогов, режимов и тревог; таблицы событий; факт и прогноз датчиков.

**Как читать оценки:**

* MAE и RMSE измеряют ошибку прогноза конкретного датчика. R² может быть отрицательным.
* Event Recall — доля аварий, перед которыми пришло уведомление в разрешённом окне.
* Notification Precision — доля уведомлений с известным исходом, которые попали перед аварией.
* Почасовой classification report оценивает **наличие подтверждённой тревоги в каждом часе**;
  положительная метка означает окно перед размеченной аварией, а не любой вид физической аномалии.
  Его Precision/Recall отличаются от событийных. Нейтральные интервалы и неизвестные остановы исключаются по правилам детектора.
* ROC AUC и Average Precision оценивают ранжирование часов по баллу отклонения. Случайный AP равен доле положительных часов,
  случайный ROC AUC — 0.5. При одном классе обе оценки не приводятся.
* Фоновые уведомления/720 ч и доля фона в тревоге показывают эксплуатационную цену обнаружения.
* Недоступные для диагностики часы не считаем отрицательным классом. Смотрите availability_operating и число ненаблюдаемых аварий.

Таблицы CSV, PNG и HTML дополнительно сохраняются в output/notebook_report и в notebook_report каждого эксперимента.
Старый расчёт может не содержать MAE/RMSE/R² или самих прогнозов — тогда это явно указано; выдуманные значения не подставляются.

In [ ]:
METRICS = DR.render_detector_results(CFG, CONTEXTS, REPORT)

## 5. Проверка конкретных участков: с авариями и без

Меняйте только период и котёл: повторного обучения не будет. Для выбора схем используйте названия из итоговой таблицы.
Графики показывают сохранённые прогнозы **модели своего месяца**, а не прогноз последней модели на старой истории.
Выберите период перед аварией из таблицы событий и сопоставимый рабочий участок без аварии — при близкой нагрузке и составе оборудования.
По обоим участкам видны ошибки, тревоги, пропуски диагностики и факт/прогноз выбранных датчиков.

По умолчанию фильтры ниже совпадают с REPORT, поэтому ячейка готова к выполнению без заполнения примеров дат.

In [ ]:
SEGMENT = {
    **REPORT,
    # "units": ["К-5"],
    # "start": "2025-05-01", "end": "2025-05-14 23:00",
    # "sensors": ["точное название датчика из таблицы"],
    "views": ["mean::instant"],
}
SEGMENT_METRICS = DR.render_detector_results(CFG, CONTEXTS, SEGMENT)

## 6. Сохранённые модели и вопрос о классификаторе

`models.csv` — список экспортов; model — текущая модель, model_history — месячные версии, checkpoints — состояния для продолжения.
Не удаляйте их, если планируете продолжать обучение или использовать модель на дашборде.

Обученный классификатор из 05 можно смотреть на знакомых участках, но это только разбор поведения.
Для оценки качества используйте его **test_predictions.parquet** для отложенных тестовых периодов:
они не участвовали в обучении и подборе порога соответствующей модели. Порог выбирается на validation, затем фиксируется для test.
В 05 оценки на размеченных предаварийных и фоновых часах уже показаны вместе с графиками.

Если классификатор обучен на всей интересующей истории, честного теста на этой же истории нет:
нужны новые более поздние наблюдения либо повторное обучение с исторической границей и отложенным будущим периодом.
Просмотр модели на её обучении не доказывает способность предупреждать будущие аварии.
Классификатор из 05 использует исходные датчики и свои окна; в 02 его оценки не подмешиваются в тревоги регрессоров.

In [ ]:
MODELS = DR.read_csv(Path(CFG["output"])/"models.csv")
DR.show_html(DR.table("Экспорты моделей для дашборда", MODELS))
print("Общий отчёт:", (Path(CFG["output"])/"notebook_report").resolve())

## 7. Только просмотр готового классификатора из 05

Обучение не запускается, модель даже не загружается: используются сохранённые тестовые баллы.
Если 05 ещё не обучен или результаты находятся в другом каталоге, появится понятное сообщение.
Задайте `root` равным OUTPUT из 05 либо `file` — конкретный test_predictions.parquet.

`folds="latest"` показывает последний тест каждого варианта, `"all"` — все.
`threshold_rule="best_f2"` использует сохранённый порог, выбранный в 05 на validation.
Если такого порога нет, применяется 0.5 с явной подписью. `units`, `start`, `end` ограничивают просмотр.
На графике видны баллы и предаварийные интервалы; распределения баллов по двум классам помогают увидеть,
различает ли модель предаварийные и фоновые часы. Classification report и confusion matrix считаются
только по часам с известным исходом, для которых модель уже была доступна.
Эти оценки относятся к выбранному периоду/модели и не усредняются с оценками регрессоров.

In [ ]:
CLASSIFIER_VIEW = {
    "root": "runs/direct_classifier_comparison",  # OUTPUT из 05
    "file": None,  # можно указать полный путь к test_predictions.parquet
    "folds": "latest",  # "all" — все тестовые периоды
    "threshold_rule": "best_f2",
    "units": None, "start": None, "end": None,
}
CLASSIFIER_TEST_METRICS = DR.saved_classifier_report(CLASSIFIER_VIEW, CFG["output"])